# Can a client be reconstructed when the dataset has no customer id?

> **Provenance.** The `card1 + addr1 + D1n` reconstruction is from Deotte and Yakovlev. See ATTRIBUTION.md.

*Setup cells below are carried from the shared analysis so this notebook runs on its own.*


In [1]:
import polars as pl
from IPython.display import Markdown, display

t = pl.scan_csv("../../kaggle/raw/train_transaction.csv")
i = pl.scan_csv("../../kaggle/raw/train_identity.csv")
df = t.join(i, on="TransactionID", how="left")
ctx = pl.SQLContext(df=df)


13,553 cards across 590,540 transactions is roughly 44 transactions per card — dense
enough for velocity features to have something to count. It is also exactly the number of
rows whose `seconds_since_prev_txn_card` is null, which is the consistency check that
proves the point-in-time frames mean what they claim.

### Reconstructing the client

No column names a client, but §2's labelling rule makes the client the natural unit of
this problem: a chargeback marks every later linked transaction as fraud. `D1` is a
timedelta in days since the client's first transaction, so `day − D1` recovers that
client's account-start day — a constant across their history. Combined with `card1` and
`addr1` it gives a client id.

The test of whether it works is **label purity**: if the id really identifies a client,
then because fraud propagates across a client's transactions, its groups should be far
more label-homogeneous than card groups.

Purity is reported on two denominators because only one of them means anything. Singleton
entities are pure **by definition** — one transaction cannot disagree with itself — so a
figure computed over all entities partly measures how much a key fragmented, and it rises
as the reconstruction gets _worse_. The column to read is the one restricted to entities
with at least two transactions:

| Entity          |    Entities | With >=2 txns | **Pure (>=2 txns)** | Pure (all) | Fraud share when the group contains any fraud |
| --------------- | ----------: | ------------: | ------------------: | ---------: | --------------------------------------------: |
| `card1`         |      13,553 |        10,109 |               84.8% |      88.7% |                                         24.9% |
| `card1 + addr1` |      37,531 |        22,713 |               89.6% |      93.7% |                                         36.3% |
| **client uid**  | **199,070** |    **83,557** |           **98.5%** |      99.4% |                                     **84.9%** |

Measured with [`techniques/entity_purity`](techniques/entity-purity.md), which reports both
denominators so they cannot be swapped for each other.

A group containing any fraud is **85% fraud** under the client id, against 25% under the card. That is the labelling rule showing up in the data, and it is the strongest evidence available that the reconstruction is real rather than arbitrary fragmentation.

Where it breaks, and why the id is NULL rather than filled in. 

Splitting by whether the components were present:

| Subset                                 | Entities | Label-pure groups |
| -------------------------------------- | -------- | ----------------- |
| `addr1` and `D1` both present          | 198,836  | **98.5%**         |
| either missing, filled with a sentinel | 18,899   | **79.4%**         |

The sentinel version is _worse than `card1`_: filling a missing address with a placeholder merges unrelated clients into one group. So the id is NULL on those 66,794 rows (11.3%) and every client aggregate is null-guarded — the same conclusion the device feature reached, arrived at the same way.



In [2]:
total_rows = df.select(pl.len()).collect().item()

ent_stats = df.select([
    pl.col("card1").n_unique().alias("card1_dist"),
    pl.col("card1").is_not_null().sum().alias("card1_cov"),
    pl.col("DeviceInfo").n_unique().alias("dev_dist"),
    pl.col("DeviceInfo").is_not_null().sum().alias("dev_cov"),
    pl.col("ProductCD").n_unique().alias("prod_dist"),
    pl.col("ProductCD").is_not_null().sum().alias("prod_cov")
]).collect().row(0)

ent_data = [
    {"Entity": "`card1`", "Distinct values": f"{ent_stats[0]:,}", "Coverage": f"{ent_stats[1]/total_rows*100:.1f}%"},
    {"Entity": "`DeviceInfo`", "Distinct values": f"{ent_stats[2]:,}", "Coverage": f"{ent_stats[3]/total_rows*100:.1f}% ({ent_stats[3]:,} rows)"},
    {"Entity": "`ProductCD`", "Distinct values": f"{ent_stats[4]:,}", "Coverage": f"{ent_stats[5]/total_rows*100:.1f}%"}
]
display(Markdown(pl.DataFrame(ent_data).to_pandas().to_markdown(index=False)))


| Entity       |   Distinct values | Coverage             |
|:-------------|------------------:|:---------------------|
| `card1`      |            13,553 | 100.0%               |
| `DeviceInfo` |             1,787 | 20.1% (118,666 rows) |
| `ProductCD`  |                 5 | 100.0%               |